[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fedgutcor/lab-ia-intermedio/blob/main/recetas/02_texto_a_numeros.ipynb)

# 02 · Del texto a los números: clasificar mensajes

**M2 · Procesamiento de lenguaje natural** · Laboratorio de IA · nivel intermedio

En esta receta vas a:
- convertir texto en tokens y vectores con `TextVectorization`
- entrenar un clasificador de mensajes con `Embedding`
- descubrir por qué una partición mal hecha infla el resultado
- revisar casos difíciles: negaciones y mensajes ambiguos

**Cómo usarla:** en Colab elige *Entorno de ejecución → Ejecutar todo*. No hace falta instalar nada. Si Colab avisa que el notebook no es de Google, elige *Ejecutar de todos modos*. Cuando termine, busca la celda **«Cambia esto»**, modifica un valor y vuelve a ejecutar.

In [ ]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import GroupShuffleSplit, train_test_split
def como_texto(x):
    # Keras 3 recibe el texto como tensor de TensorFlow, no como columna de pandas
    return tf.constant([str(t) for t in x])

INICIO = time.time()
DATOS = "https://raw.githubusercontent.com/Fedgutcor/lab-ia-intermedio/main/datos/mensajes_soporte.csv"
df = pd.read_csv(DATOS)
print(len(df), "mensajes")
df.sample(6, random_state=1)

## 1 · Conoce los datos

Son **mensajes sintéticos** (generados por combinación de frases, sin datos de personas). Cada uno tiene su motivo real, su dificultad (`normal`, `negación`, `ambiguo`) y dos columnas para agruparlos:

- `grupo`: los **casi-duplicados**, el mismo mensaje con otro saludo, otro producto o sin tildes;
- `frase`: la **intención de fondo**, por ejemplo «me cobraron dos veces».

In [ ]:
print(df["motivo"].value_counts(), "\n")
print(df["dificultad"].value_counts())
CLASES = sorted(df["motivo"].unique())
df["y"] = df["motivo"].map({c: i for i, c in enumerate(CLASES)})

## 2 · Texto → tokens → números

`TextVectorization` arma un vocabulario y convierte cada mensaje en una lista de números. Mira qué le pasa a un mensaje con negación. (Este vocabulario es solo para mirar: más abajo cada modelo aprende el suyo únicamente con sus datos de entrenamiento.)

In [ ]:
vectorizar = layers.TextVectorization(max_tokens=3000, output_sequence_length=24)
vectorizar.adapt(df["mensaje"].values)
vocabulario = vectorizar.get_vocabulary()
print("Tamaño del vocabulario:", len(vocabulario))
print("Primeras palabras:", vocabulario[2:22])

ejemplo = df[df["dificultad"] == "negación"]["mensaje"].iloc[0]
ids = vectorizar([ejemplo])[0].numpy()
print("\nMensaje:", ejemplo)
print("Números:", ids)
print("De vuelta a palabras:", [vocabulario[i] for i in ids if i > 0])

## 3 · Cambia esto

In [ ]:
DIMENSION = 16      # tamaño de cada vector de palabra (prueba 4 o 64)
EPOCAS = 30
TASA = 0.01         # tasa de aprendizaje (prueba 0.001: con pocas épocas el modelo queda a medias)

## 4 · Tres formas de partir los datos

Se entrena el mismo modelo con tres particiones distintas:

1. **Por fila:** los casi-duplicados quedan a los dos lados, así que la prueba se parece demasiado al entrenamiento.
2. **Por grupo:** ningún casi-duplicado se repite. Es la evaluación honesta para mensajes como los que ya conoces.
3. **Por frase:** la prueba trae formas de pedir que el modelo nunca vio. Es la prueba más exigente.

Antes de ejecutar, predice: ¿cuál dará el número más alto y cuál el más bajo?

In [ ]:
def crear_modelo(textos_entreno):
    # el vocabulario se aprende SOLO del entrenamiento: si viera la prueba, la evaluación ya no sería limpia
    vec = layers.TextVectorization(max_tokens=3000, output_sequence_length=24)
    vec.adapt(textos_entreno)
    m = keras.Sequential([
        keras.Input((1,), dtype="string"),
        vec,
        layers.Embedding(vec.vocabulary_size(), DIMENSION),
        layers.GlobalAveragePooling1D(),
        layers.Dense(len(CLASES), activation="softmax"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(TASA), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def entrenar_y_medir(entreno, prueba):
    m = crear_modelo(entreno["mensaje"].values)
    m.fit(como_texto(entreno["mensaje"]), entreno["y"].values, epochs=EPOCAS, batch_size=32, verbose=0)
    return m, m.evaluate(como_texto(prueba["mensaje"]), prueba["y"].values, verbose=0)[1]

ent_fila, pru_fila = train_test_split(df, test_size=0.25, random_state=0, stratify=df["y"])
_, exactitud_fila = entrenar_y_medir(ent_fila, pru_fila)

def por_columna(columna):
    i_ent, i_pru = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(df, groups=df[columna]))
    return df.iloc[i_ent], df.iloc[i_pru]

_, exactitud_grupo = entrenar_y_medir(*por_columna("grupo"))
ent_fr, pru_fr = por_columna("frase")
modelo, exactitud_frase = entrenar_y_medir(ent_fr, pru_fr)

print(f"Por fila:  {exactitud_fila:.3f}   (casi-duplicados a los dos lados)")
print(f"Por grupo: {exactitud_grupo:.3f}   (sin casi-duplicados compartidos)")
print(f"Por frase: {exactitud_frase:.3f}   (formas de pedir nunca vistas)")

**Cómo leerlo.** Si por fila y por grupo dan casi lo mismo y por frase baja, el modelo está **reconociendo frases que ya vio**. Con mensajes reales llegarán formas nuevas de pedir lo mismo: el número que más se parece a eso es el de la partición por frase. Un resultado perfecto en datos sintéticos no prueba que el modelo entienda.

## 5 · ¿Dónde se equivoca?

Mira los errores de la partición por frase y sepáralos por dificultad.

In [ ]:
prueba = pru_fr.copy()
prueba["prediccion"] = [CLASES[i] for i in modelo.predict(como_texto(prueba["mensaje"]), verbose=0).argmax(axis=1)]
prueba["acierto"] = prueba["prediccion"] == prueba["motivo"]
print(prueba.groupby("dificultad")["acierto"].agg(["mean", "count"]).round(2), "\n")
pd.set_option("display.max_colwidth", 90)
prueba[~prueba["acierto"]][["mensaje", "motivo", "prediccion", "dificultad"]].head(10)

## 6 · Prueba tus propios mensajes

In [ ]:
MIS_MENSAJES = [
    "No me llega el pedido y ya pasó una semana",
    "Quiero que me devuelvan la plata",
    "No quiero devolver nada, solo cambiar la contraseña",
]
for texto, p in zip(MIS_MENSAJES, modelo.predict(como_texto(MIS_MENSAJES), verbose=0)):
    print(f"{CLASES[p.argmax()]:<20} ({p.max():.2f})  {texto}")
print(f"\nTiempo total de la receta: {time.time() - INICIO:.0f} s")

## Para tu proyecto

- Si tus datos tienen casi-duplicados (mensajes de la misma persona, fotos del mismo objeto), **parte por grupo**.
- Un modelo así no entiende la negación: la aprende solo si hay ejemplos suficientes. Revisa siempre los casos difíciles por separado.